# Feature Engineering for supply-anomaly-classification

This notebook builds a compact and interpretable feature set for order-level anomaly detection using the available transactional data.

### Main goals
- Derive operational and financial signals from the raw order table
- Capture shipping, pricing, discount, and calendar deviations
- Keep the feature set interpretable and leakage-safe
- Prepare the dataset for downstream anomaly modeling

In [1]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler

from pathlib import Path
from config.settings import EXPORT_DIR, DATA_DIR

In [2]:
export_dir = Path(EXPORT_DIR)
data_dir = Path(DATA_DIR)

---
## Load prepared datasets

In [3]:
# Load dataco
df = pd.read_parquet(export_dir / 'dataco_df.parquet')

In [4]:
print('df', df.shape)
df.head()


df (171962, 12)


,Days for shipping (real),Days for shipment (scheduled),Department Name,Market,order date (DateOrders),Order Item Discount,Order Item Product Price,Order Item Quantity,Order Item Total,Order Status,Product Category Id,Shipping Mode
0,2,4,Fan Shop,LATAM,2015-01-01 00:00:00,60.0,299.980011,1,239.980011,COMPLETE,43,Standard Class
1,3,4,Fan Shop,LATAM,2015-01-01 00:21:00,6.0,199.990005,1,193.990005,PENDING_PAYMENT,48,Standard Class
2,3,4,Apparel,LATAM,2015-01-01 00:21:00,22.1,129.990005,1,107.890007,PENDING_PAYMENT,18,Standard Class
3,3,4,Golf,LATAM,2015-01-01 00:21:00,22.5,50.000000,5,137.500000,PENDING_PAYMENT,24,Standard Class
4,5,4,Apparel,LATAM,2015-01-01 01:03:00,3.0,59.990002,5,284.950012,COMPLETE,17,Standard Class


---
## Shipping features

Shipping-related features capture delivery delays, service level differences, and coarse shipping-speed categories.

In [5]:
df['shipping_delay'] = df['Days for shipping (real)'] - df['Days for shipment (scheduled)']
df['is_delay'] = (df['shipping_delay'] > 0).astype('int8')
df['delay_severity'] = (df['shipping_delay'].abs() / (df['Days for shipment (scheduled)'] + 1e-5)).clip(0, 1)

In [6]:
# Shipping speed categories
conditions = [
    df['Days for shipping (real)'] <= 1,
    df['Days for shipping (real)'] <= 3,
    df['Days for shipping (real)'] <= 5
]

df['shipping_speed'] = np.select(conditions, ['express', 'fast', 'standard'], default='slow')

---
## Price/Discount variance features

These features measure how far each order deviates from its local price and discount benchmark.

In [7]:
# Price benchmarks by product category and market
df['price_variance_abs'] = (df['Order Item Product Price'] - 
                            df.groupby(['Product Category Id', 'Market'], observed=True)['Order Item Product Price'].transform('mean')
                           ).abs()

In [8]:
# Discount benchmarks by department and product category
df['discount_variance_abs'] = (df['Order Item Discount'] - 
                               df.groupby(['Department Name', 'Product Category Id'], observed=True)['Order Item Discount'].transform('mean')
                              ).abs()

---
## Calendar features
- week and month are encoded as sine/cosine pairs instead of raw integers or one-hot dummies
- this avoids the false "distance" a model would otherwise see between, e.g., 1 and 53, or December and January

In [9]:
week = df['order date (DateOrders)'].dt.isocalendar().week.astype('int16')
month = df['order date (DateOrders)'].dt.month

df['week_sin'] = np.sin(2 * np.pi * week / 53)
df['week_cos'] = np.cos(2 * np.pi * week / 53)

df['month_sin'] = np.sin(2 * np.pi * month / 12)
df['month_cos'] = np.cos(2 * np.pi * month / 12)

df['is_peak_period'] = month.isin([11, 12, 1]).astype('int8')

---
## Severity score
Severity score combines the strongest anomaly signals into a single continuous proxy for downstream ranking.

In [10]:
features_for_score = ['price_variance_abs', 'discount_variance_abs', 'delay_severity']

scaler = MinMaxScaler()

In [11]:
features_scaled = scaler.fit_transform(df[features_for_score])

df['severity_score'] = (0.4 * features_scaled[:, 0] + 
                        0.25 * features_scaled[:, 1] +
                        0.35 * features_scaled[:, 2])

---
## Feature audit
A final audit helps verify data types, memory usage, and the overall shape of the engineered dataset.

In [12]:
feature_summary = pd.DataFrame({
    'dtype': df.dtypes,
    'missing_pct': df.isna().mean() * 100,
    'n_unique': df.nunique()
}).sort_values('missing_pct', ascending=False)

feature_summary

,dtype,missing_pct,n_unique
Days for shipping (real),int8,0.0,7
Days for shipment (scheduled),int8,0.0,4
Department Name,category,0.0,6
Market,category,0.0,5
order date (DateOrders),datetime64[ns],0.0,57349
Order Item Discount,float32,0.0,726
Order Item Product Price,float32,0.0,57
Order Item Quantity,int8,0.0,5
Order Item Total,float32,0.0,2309
Order Status,category,0.0,6


In [13]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Days for shipping (real),171962.0,3.497249,0.0,2.0,3.0,5.0,6.0,1.623859
Days for shipment (scheduled),171962.0,2.931944,0.0,2.0,4.0,4.0,4.0,1.37446
order date (DateOrders),171962,2016-05-17 01:52:21.930310400,2015-01-01 00:00:00,2015-09-08 21:48:15,2016-05-17 09:27:00,2017-01-23 09:16:00,2017-09-30 23:59:00,NaN
Order Item Discount,171962.0,19.914879,-0.0,5.76,14.0,29.99,500.0,19.223717
Order Item Product Price,171962.0,133.594696,9.99,50.0,59.990002,199.990005,1999.98999,117.608238
Order Item Quantity,171962.0,2.182383,1.0,1.0,1.0,3.0,5.0,1.46642
Order Item Total,171962.0,153.802795,9.99,75.0,123.490005,199.950012,1699.98999,101.833534
Product Category Id,171962.0,30.099208,2.0,18.0,29.0,45.0,48.0,13.716604
shipping_delay,171962.0,0.565305,-2.0,0.0,1.0,1.0,4.0,1.490738
is_delay,171962.0,0.572615,0.0,0.0,1.0,1.0,1.0,0.4947


In [14]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 171962 entries, 0 to 171961
Data columns (total 24 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   Days for shipping (real)       171962 non-null  int8          
 1   Days for shipment (scheduled)  171962 non-null  int8          
 2   Department Name                171962 non-null  category      
 3   Market                         171962 non-null  category      
 4   order date (DateOrders)        171962 non-null  datetime64[ns]
 5   Order Item Discount            171962 non-null  float32       
 6   Order Item Product Price       171962 non-null  float32       
 7   Order Item Quantity            171962 non-null  int8          
 8   Order Item Total               171962 non-null  float32       
 9   Order Status                   171962 non-null  category      
 10  Product Category Id            171962 non-null  int8          
 11  

In [15]:
float_cols = ['delay_severity', 'price_variance_abs', 'discount_variance_abs', 
              'week_sin', 'week_cos', 'month_sin', 'month_cos', 'severity_score']
cat_cols = ['shipping_speed']

df[float_cols] = df[float_cols].astype('float32')
df[cat_cols] = df[cat_cols].astype('category')

In [16]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
Days for shipping (real),171962.0,3.497249,0.0,2.0,3.0,5.0,6.0,1.623859
Days for shipment (scheduled),171962.0,2.931944,0.0,2.0,4.0,4.0,4.0,1.37446
order date (DateOrders),171962,2016-05-17 01:52:21.930310400,2015-01-01 00:00:00,2015-09-08 21:48:15,2016-05-17 09:27:00,2017-01-23 09:16:00,2017-09-30 23:59:00,NaN
Order Item Discount,171962.0,19.914879,-0.0,5.76,14.0,29.99,500.0,19.223717
Order Item Product Price,171962.0,133.594696,9.99,50.0,59.990002,199.990005,1999.98999,117.608238
Order Item Quantity,171962.0,2.182383,1.0,1.0,1.0,3.0,5.0,1.46642
Order Item Total,171962.0,153.802795,9.99,75.0,123.490005,199.950012,1699.98999,101.833534
Product Category Id,171962.0,30.099208,2.0,18.0,29.0,45.0,48.0,13.716604
shipping_delay,171962.0,0.565305,-2.0,0.0,1.0,1.0,4.0,1.490738
is_delay,171962.0,0.572615,0.0,0.0,1.0,1.0,1.0,0.4947


In [17]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 171962 entries, 0 to 171961
Data columns (total 24 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   Days for shipping (real)       171962 non-null  int8          
 1   Days for shipment (scheduled)  171962 non-null  int8          
 2   Department Name                171962 non-null  category      
 3   Market                         171962 non-null  category      
 4   order date (DateOrders)        171962 non-null  datetime64[ns]
 5   Order Item Discount            171962 non-null  float32       
 6   Order Item Product Price       171962 non-null  float32       
 7   Order Item Quantity            171962 non-null  int8          
 8   Order Item Total               171962 non-null  float32       
 9   Order Status                   171962 non-null  category      
 10  Product Category Id            171962 non-null  int8          
 11  

In [18]:
# Quick correlation check against the secondary target
df.corr(numeric_only=True)['severity_score'].sort_values(ascending=False)

severity_score                   1.000000
delay_severity                   0.998316
is_delay                         0.666320
shipping_delay                   0.487165
discount_variance_abs            0.048474
price_variance_abs               0.037273
Order Item Discount              0.024831
Order Item Product Price         0.017394
Order Item Total                 0.012601
Order Item Quantity              0.004007
Product Category Id              0.002524
month_cos                       -0.004343
week_sin                        -0.005154
week_cos                        -0.005378
month_sin                       -0.005454
is_peak_period                  -0.007742
Days for shipping (real)        -0.066688
Days for shipment (scheduled)   -0.607167
Name: severity_score, dtype: float64

In [19]:
# Export feature-engineered dataset
df.to_parquet(data_dir / 'features_df.parquet')
print('Feature engineered dataset saved.')

Feature engineered dataset saved.


## Key feature engineering takeaways

- Shipping delay is one of the clearest operational anomaly signals
- Price and discount variances are more informative than raw values alone
- Cyclical calendar encoding preserves seasonality without creating false distance